# 02 — Preprocessing Pipeline

Steps covered:
1. Data loading and missing value imputation
2. Sliding window segmentation (5 s window, 2 s overlap → 3 s stride)
3. Full feature extraction (statistical + FFT + derived)
4. Save processed features

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_loader import UAHDriveSetLoader
from preprocessing import (
    fill_missing,
    segment_trip,
    process_trips,
    scale_features,
    DEFAULT_STAT_COLS,
    DEFAULT_FFT_COLS,
)

DATA_DIR  = Path('../data/raw/UAH-DRIVESET-v1')
PROC_DIR  = Path('../data/processed'); PROC_DIR.mkdir(exist_ok=True)
FIG_DIR   = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_S  = 5.0   # seconds
OVERLAP_S = 2.0   # seconds overlap

## 1. Data Loading

In [ ]:
loader = UAHDriveSetLoader(DATA_DIR, merge=True).load(verbose=True)
print(f'Trips loaded: {len(loader)}')

## 2. Missing Value Analysis

In [ ]:
trip = loader.trips[0]
df_raw = trip.merged if not trip.merged.empty else trip.accel

missing_pct = df_raw.isnull().mean() * 100
print('Missing value percentage (first trip):')
print(missing_pct[missing_pct > 0].sort_values(ascending=False).to_string())

df_filled = fill_missing(df_raw)
print(f'\nRemaining NaN after imputation: {df_filled.isnull().sum().sum()}')

## 3. Window Segmentation Visualisation

In [ ]:
segments = segment_trip(df_filled, window_s=WINDOW_S, overlap_s=OVERLAP_S)
print(f'Number of windows: {len(segments)}')
print(f'Average window length: {np.mean([len(s) for s in segments]):.1f} rows')

# Visualise the first 3 windows
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_filled['timestamp'], df_filled['acc_x_kf'], color='gray', alpha=0.4, label='Full signal')
colors = ['#e41a1c', '#377eb8', '#4daf4a']
for i, seg in enumerate(segments[:3]):
    ax.plot(seg['timestamp'], seg['acc_x_kf'], color=colors[i], linewidth=2, label=f'Window {i+1}')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Acc X KF (G)')
ax.set_title(f'Sliding Window Segmentation ({WINDOW_S}s window, {OVERLAP_S}s overlap)')
ax.legend(); plt.tight_layout()
fig.savefig(FIG_DIR / 'preprocessing_windows.png', dpi=150)
plt.show()

## 4. Full Feature Extraction

In [ ]:
df_features = process_trips(
    loader.trips,
    window_s=WINDOW_S,
    overlap_s=OVERLAP_S,
    verbose=True,
)
print(f'\nFeature table shape: {df_features.shape}')
print(f'Number of windows: {len(df_features)}')
print(f'Number of features: {df_features.shape[1] - 6}')  # excluding meta columns
df_features.head(3)

In [ ]:
# Class distribution
print('Class distribution:')
print(df_features['behavior'].value_counts().to_string())

## 5. Save

In [ ]:
out_path = PROC_DIR / 'features_5s_2s.parquet'
df_features.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')

# CSV backup
df_features.to_csv(PROC_DIR / 'features_5s_2s.csv', index=False)
print('CSV backup also saved.')

## 6. Feature Distribution Check

In [ ]:
# Box plot for a few key features
key_feats = ['acc_x_kf_mean', 'acc_y_kf_mean', 'acc_x_kf_std', 'jerk_x_mean', 'lateral_g_mean']
avail = [f for f in key_feats if f in df_features.columns]

fig, axes = plt.subplots(1, len(avail), figsize=(4 * len(avail), 5))
if len(avail) == 1:
    axes = [axes]

palette = {'normal': '#4C72B0', 'aggressive': '#DD8452', 'economic': '#55A868'}
for ax, feat in zip(axes, avail):
    sns.boxplot(
        data=df_features, x='behavior', y=feat,
        palette=palette, ax=ax, order=['normal', 'aggressive', 'economic']
    )
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')

plt.suptitle('Key Feature Distributions by Behavior')
plt.tight_layout()
fig.savefig(FIG_DIR / 'preprocessing_feature_boxplots.png', dpi=150)
plt.show()